<div align="center">

# 📥 Thu thập dữ liệu (Data Collection)

</div>


### Import thư viện 📦

In [1]:

# import các thư viện cần thiết
from tvDatafeed import TvDatafeed, Interval
import ccxt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests as rq
import warnings
warnings.filterwarnings("ignore")



### Định nghĩa hàm sử dụng ⚙️

In [2]:

# Thu thập dữ liệu về giá của Bitcoin từ API của Binance
def Crawl_Bitcoin_price(object_binace,symbol,timeframe,limit,since):
    bars = object_binace.fetch_ohlcv(
        symbol,
        timeframe=timeframe,
        since=since,
        limit=limit
    )
    return bars


### Thu thập dữ liệu giá của Bitcoin từ 1-1-2018 đến 7-3-2026 (Nhóm dữ liệu bắt buộc OHLCV) 🪙

In [3]:

# Khởi tạo 1 đối tượng trong lớp Binace của thư viện ccxt bằng câu lệnh ccxt.binance()
exchange = ccxt.binance()

# Khởi tạo các tham số để thu thập dữ liệu về giá của Bitcoin
symbol = 'BTC/USDT' # mã của Bitcoin
timeframe = '1h'    # lấy tần suất theo giờ
limit = 1000        # giới hạn mỗi lần lấy 1000 sample (bời API của Binace chỉ cho lấy 1000 dữ liệu 1 lần)
since = exchange.parse8601('2018-01-01T00:00:00Z') # định dạng thời gian trong ngoặc là định dạng thời gian dạng chữ chuẩn quốc tế ISO-8601
                                                   # tuy nhiên API của Binace chỉ hiểu time dưới dạng ms nên câu lệnh này dùng để quy đổi thời gian dạng chữ sang chuẩn ms để API hiểu       

# khởi tạo 1 list rỗng để chưa data mà API Binance trả về bởi API Binance trả về dữ liệu dưới dạng ma trận 2 chiều
all_data = []

# Vòng lặp để tối ưu hóa việc lấy dữ liệu (bởi vì dự án cần trên 10000 dữ liệu mà API Binance chỉ cho lấy 1000/1 lần nên cần lặp lại)
while len(all_data) < 100000:

    bars = Crawl_Bitcoin_price(exchange,symbol,timeframe,limit,since)

    # Check kiểm tra xem bars còn trả về dữ liệu hay ko, nếu ko ngừng trương trình
    if len(bars) == 0:
        break

    all_data.extend(bars) 

    # + 1ms vào mốc thời gian mới để né cơ chế lấy mốc thời gian >= since của API Binance
    since = bars[-1][0] + 1

    # print("Collected:", len(all_data))

# Chuyển mảng hai chiều về dạng Dataframe trong pandas để dễ làm việc
df = pd.DataFrame(all_data, columns=[
    'timestamp','open','high','low','close','volume'
])

# quy đổi thời gian từ ms sang hệ chuẩn ISO-8601
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
df = df.sort_values('timestamp')

df


,timestamp,open,high,low,close,volume
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006
2,2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572
3,2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030
4,2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329
...,...,...,...,...,...,...
71661,2026-03-10 23:00:00,69721.58,69981.80,69698.46,69948.63,1165.159140
71662,2026-03-11 00:00:00,69948.64,70173.25,69759.63,70044.87,728.929610
71663,2026-03-11 01:00:00,70044.87,70144.00,69880.00,70071.26,947.248790
71664,2026-03-11 02:00:00,70071.26,70119.59,69614.00,69810.72,1612.875480


### Tính toán các dữ liệu chỉ báo kỹ thuật từ dữ liệu giá Bitcoin bên trên 📐

In [4]:
# Tính toán chỉ báo kỹ thuật ATR (Average True Range)

# Tạo dataframe data để chứa các chỉ báo kỹ thuật phụ 
data = pd.DataFrame()

data['prev_close'] = df['close'].shift(1)

data['TR1'] = df['high'] - df['low']
data['TR2'] = (df['high'] - data['prev_close']).abs()
data['TR3'] = (df['low'] - data['prev_close']).abs()

data['TR'] = data[['TR1','TR2','TR3']].max(axis=1)
df['ATR'] = data['TR'].ewm(span=336,adjust=False).mean()
df


,timestamp,open,high,low,close,volume,ATR
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,315.640000
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,316.381068
2,2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,315.799756
3,2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,315.832221
4,2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,315.745026
...,...,...,...,...,...,...,...
71661,2026-03-10 23:00:00,69721.58,69981.80,69698.46,69948.63,1165.159140,630.619773
71662,2026-03-11 00:00:00,69948.64,70173.25,69759.63,70044.87,728.929610,629.331941
71663,2026-03-11 01:00:00,70044.87,70144.00,69880.00,70071.26,947.248790,627.163799
71664,2026-03-11 02:00:00,70071.26,70119.59,69614.00,69810.72,1612.875480,626.442292


In [5]:
# Tính toán chỉ báo kỹ thuật Bollinger Bands Width

data['MA20'] = df['close'].rolling(window=480).mean()

data['SD20'] = df['close'].rolling(window=480).std(ddof=1)

data['upper'] = data['MA20'] + 2 * data['SD20']
data['lower'] = data['MA20'] - 2 * data['SD20']

df['BB_width_norm'] = (data['upper'] - data['lower'])/data['MA20']
df

,timestamp,open,high,low,close,volume,ATR,BB_width_norm
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,315.640000,NaN
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,316.381068,NaN
2,2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,315.799756,NaN
3,2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,315.832221,NaN
4,2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,315.745026,NaN
...,...,...,...,...,...,...,...,...
71661,2026-03-10 23:00:00,69721.58,69981.80,69698.46,69948.63,1165.159140,630.619773,0.125565
71662,2026-03-11 00:00:00,69948.64,70173.25,69759.63,70044.87,728.929610,629.331941,0.125669
71663,2026-03-11 01:00:00,70044.87,70144.00,69880.00,70071.26,947.248790,627.163799,0.125801
71664,2026-03-11 02:00:00,70071.26,70119.59,69614.00,69810.72,1612.875480,626.442292,0.125898


In [6]:
# Tính toán chỉ báo kỹ thuật RSI (Relative Strength Index)

n=336
data['delta'] = df['close'].diff()

data['Gain'] = data['delta'].clip(lower=0)
data['Loss'] = data['delta'].clip(upper=0)

data['AvgGain'] = data['Gain'].ewm(alpha=1/n, adjust=False).mean()
data['AvgLoss'] = data['Loss'].ewm(alpha=1/n, adjust=False).mean()

data['RS'] = data['AvgGain']/data['AvgLoss']

df['RSI'] = 100 - (100 / (1 + data['RS']))
df

,timestamp,open,high,low,close,volume,ATR,BB_width_norm,RSI
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,315.640000,NaN,NaN
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,316.381068,NaN,0.000000
2,2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,315.799756,NaN,-0.116553
3,2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,315.832221,NaN,-0.190124
4,2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,315.745026,NaN,-0.367052
...,...,...,...,...,...,...,...,...,...
71661,2026-03-10 23:00:00,69721.58,69981.80,69698.46,69948.63,1165.159140,630.619773,0.125565,22715.462812
71662,2026-03-11 00:00:00,69948.64,70173.25,69759.63,70044.87,728.929610,629.331941,0.125669,15875.054066
71663,2026-03-11 01:00:00,70044.87,70144.00,69880.00,70071.26,947.248790,627.163799,0.125801,14663.557223
71664,2026-03-11 02:00:00,70071.26,70119.59,69614.00,69810.72,1612.875480,626.442292,0.125898,61217.518220


In [7]:
# Tính chỉ số kỹ thuật MACD (Moving Average Convergence Divergence)

data['EMA12'] = df['close'].ewm(span=288,adjust=False).mean()

data['EMA26'] = df['close'].ewm(span=624,adjust=False).mean()

data['MACD'] = data['EMA12'] - data['EMA26']

data['Signal'] = data['MACD'].ewm(span=216, adjust=False).mean()

df['MACD_Hist'] = data['MACD'] - data['Signal']
df

,timestamp,open,high,low,close,volume,ATR,BB_width_norm,RSI,MACD_Hist
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,315.640000,NaN,NaN,0.000000
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,316.381068,NaN,0.000000,-1.201493
2,2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,315.799756,NaN,-0.116553,-1.911172
3,2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,315.832221,NaN,-0.190124,-2.312652
4,2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,315.745026,NaN,-0.367052,-2.002172
...,...,...,...,...,...,...,...,...,...,...
71661,2026-03-10 23:00:00,69721.58,69981.80,69698.46,69948.63,1165.159140,630.619773,0.125565,22715.462812,798.079696
71662,2026-03-11 00:00:00,69948.64,70173.25,69759.63,70044.87,728.929610,629.331941,0.125669,15875.054066,799.139303
71663,2026-03-11 01:00:00,70044.87,70144.00,69880.00,70071.26,947.248790,627.163799,0.125801,14663.557223,800.221240
71664,2026-03-11 02:00:00,70071.26,70119.59,69614.00,69810.72,1612.875480,626.442292,0.125898,61217.518220,800.267129


In [8]:

# Tính chỉ số kỹ thuật MA20 (Mean Average 20)

df['MA20'] = df['close'].rolling(window=480).mean()
df


,timestamp,open,high,low,close,volume,ATR,BB_width_norm,RSI,MACD_Hist,MA20
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,315.640000,NaN,NaN,0.000000,NaN
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,316.381068,NaN,0.000000,-1.201493,NaN
2,2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,315.799756,NaN,-0.116553,-1.911172,NaN
3,2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,315.832221,NaN,-0.190124,-2.312652,NaN
4,2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,315.745026,NaN,-0.367052,-2.002172,NaN
...,...,...,...,...,...,...,...,...,...,...,...
71661,2026-03-10 23:00:00,69721.58,69981.80,69698.46,69948.63,1165.159140,630.619773,0.125565,22715.462812,798.079696,67678.468042
71662,2026-03-11 00:00:00,69948.64,70173.25,69759.63,70044.87,728.929610,629.331941,0.125669,15875.054066,799.139303,67685.964667
71663,2026-03-11 01:00:00,70044.87,70144.00,69880.00,70071.26,947.248790,627.163799,0.125801,14663.557223,800.221240,67692.641000
71664,2026-03-11 02:00:00,70071.26,70119.59,69614.00,69810.72,1612.875480,626.442292,0.125898,61217.518220,800.267129,67698.840000


In [9]:

# Tính chỉ số kỹ thuật EMA20 (Exponential Moving Average)
df['EMA20'] = df['close'].ewm(span=480,adjust=False).mean()
df


,timestamp,open,high,low,close,volume,ATR,BB_width_norm,RSI,MACD_Hist,MA20,EMA20
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,315.640000,NaN,NaN,0.000000,NaN,13529.010000
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,316.381068,NaN,0.000000,-1.201493,NaN,13527.654699
2,2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,315.799756,NaN,-0.116553,-1.911172,NaN,13526.833598
3,2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,315.832221,NaN,-0.190124,-2.312652,NaN,13526.347928
4,2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,315.745026,NaN,-0.367052,-2.002172,NaN,13526.658373
...,...,...,...,...,...,...,...,...,...,...,...,...
71661,2026-03-10 23:00:00,69721.58,69981.80,69698.46,69948.63,1165.159140,630.619773,0.125565,22715.462812,798.079696,67678.468042,68793.166000
71662,2026-03-11 00:00:00,69948.64,70173.25,69759.63,70044.87,728.929610,629.331941,0.125669,15875.054066,799.139303,67685.964667,68798.370590
71663,2026-03-11 01:00:00,70044.87,70144.00,69880.00,70071.26,947.248790,627.163799,0.125801,14663.557223,800.221240,67692.641000,68803.663270
71664,2026-03-11 02:00:00,70071.26,70119.59,69614.00,69810.72,1612.875480,626.442292,0.125898,61217.518220,800.267129,67698.840000,68807.850616


### Thu thập các dữ liệu thị trường chung ảnh hưởng đến giá Bitcoin (CFGI, BTCD, USDTD, Total CMC, Single CMC) 📈

In [10]:

# Thu thập dữ liệu về CFGI (Crypto Fear & Greed index)

# Khởi tạo biến chứa địa chỉ API để lấy dữ liệu
url = "https://api.alternative.me/fng/?limit=0"

# sử dụng thư viện requests để gửi truy vấn đến API
Json_api = rq.get(url).json()

# Dữ liệu được trả về từ API dưới dạng json, chuyển đổi nó sang dạng dataframe
CFGI = pd.DataFrame(Json_api["data"])

# Chuyển đổi từ timestamp sang datetime
CFGI['timestamp'] = pd.to_datetime(CFGI['timestamp'], unit='s');

# Chuyển đổi dữ liệu từ string sang int
CFGI['value'] = CFGI['value'].astype(int)

# xóa đi các cột không cần thiết
CFGI = CFGI[['timestamp','value']]
CFGI = CFGI.sort_values('timestamp')
CFGI.rename(columns={'value':'CFGI'},inplace=True)
CFGI

,timestamp,CFGI
2956,2018-02-01,30
2955,2018-02-02,15
2954,2018-02-03,40
2953,2018-02-04,24
2952,2018-02-05,11
...,...,...
4,2026-03-07,12
3,2026-03-08,12
2,2026-03-09,8
1,2026-03-10,13


In [11]:

# Tạo định dạng date cho df để kết hợp với data CFGI
df['Date'] = df['timestamp'].dt.date
CFGI['Date'] = CFGI['timestamp'].dt.date

df

,timestamp,open,high,low,close,volume,ATR,BB_width_norm,RSI,MACD_Hist,MA20,EMA20,Date
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,315.640000,NaN,NaN,0.000000,NaN,13529.010000,2018-01-01
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,316.381068,NaN,0.000000,-1.201493,NaN,13527.654699,2018-01-01
2,2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,315.799756,NaN,-0.116553,-1.911172,NaN,13526.833598,2018-01-01
3,2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,315.832221,NaN,-0.190124,-2.312652,NaN,13526.347928,2018-01-01
4,2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,315.745026,NaN,-0.367052,-2.002172,NaN,13526.658373,2018-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
71661,2026-03-10 23:00:00,69721.58,69981.80,69698.46,69948.63,1165.159140,630.619773,0.125565,22715.462812,798.079696,67678.468042,68793.166000,2026-03-10
71662,2026-03-11 00:00:00,69948.64,70173.25,69759.63,70044.87,728.929610,629.331941,0.125669,15875.054066,799.139303,67685.964667,68798.370590,2026-03-11
71663,2026-03-11 01:00:00,70044.87,70144.00,69880.00,70071.26,947.248790,627.163799,0.125801,14663.557223,800.221240,67692.641000,68803.663270,2026-03-11
71664,2026-03-11 02:00:00,70071.26,70119.59,69614.00,69810.72,1612.875480,626.442292,0.125898,61217.518220,800.267129,67698.840000,68807.850616,2026-03-11


In [12]:

# Kết hợp dữ liệu giá bán và dữ liệu thị trường CFGI
df = df.merge(
    CFGI[['Date','CFGI']],
    left_on = 'Date',
    right_on = 'Date',
    how='left'
)

df


,timestamp,open,high,low,close,volume,ATR,BB_width_norm,RSI,MACD_Hist,MA20,EMA20,Date,CFGI
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,315.640000,NaN,NaN,0.000000,NaN,13529.010000,2018-01-01,NaN
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,316.381068,NaN,0.000000,-1.201493,NaN,13527.654699,2018-01-01,NaN
2,2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,315.799756,NaN,-0.116553,-1.911172,NaN,13526.833598,2018-01-01,NaN
3,2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,315.832221,NaN,-0.190124,-2.312652,NaN,13526.347928,2018-01-01,NaN
4,2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,315.745026,NaN,-0.367052,-2.002172,NaN,13526.658373,2018-01-01,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71661,2026-03-10 23:00:00,69721.58,69981.80,69698.46,69948.63,1165.159140,630.619773,0.125565,22715.462812,798.079696,67678.468042,68793.166000,2026-03-10,13.0
71662,2026-03-11 00:00:00,69948.64,70173.25,69759.63,70044.87,728.929610,629.331941,0.125669,15875.054066,799.139303,67685.964667,68798.370590,2026-03-11,15.0
71663,2026-03-11 01:00:00,70044.87,70144.00,69880.00,70071.26,947.248790,627.163799,0.125801,14663.557223,800.221240,67692.641000,68803.663270,2026-03-11,15.0
71664,2026-03-11 02:00:00,70071.26,70119.59,69614.00,69810.72,1612.875480,626.442292,0.125898,61217.518220,800.267129,67698.840000,68807.850616,2026-03-11,15.0


In [13]:

# Thu thập dữ liệu BTCD (Bitcoin Dominace) bằng thư viện Tvdatafeed từ nền tảng biểu đồ Trading view

# Khởi tạo 1 đối tượng trong thư viện Tvdatafeed()
tv = TvDatafeed()

# Sử dụng hàm get_hist để lấy dữ liệu lịch sử
BTCD = tv.get_hist(
    symbol='BTC.D',             # Mã feature mà mình muốn lấy
    exchange='CRYPTOCAP',       # Nguồn lấy dữ liệu vốn hóa thị trường trên Trading View
    interval=Interval.in_daily, # Lấy tần suất dữ liệu theo ngày
    n_bars=5000
)

BTCD


you are using nologin method, data you access may be limited


,symbol,open,high,low,close,volume
datetime,,,,,,
2014-04-01 07:00:00,CRYPTOCAP:BTC.D,99.380551,99.426704,99.296832,99.384652,1.493888e+07
2014-04-02 07:00:00,CRYPTOCAP:BTC.D,99.338089,99.433992,99.332664,99.338387,1.493888e+07
2014-04-03 07:00:00,CRYPTOCAP:BTC.D,99.339295,99.494693,99.320638,99.399426,1.493888e+07
2014-04-04 07:00:00,CRYPTOCAP:BTC.D,99.409834,99.447563,99.384197,99.410501,1.493888e+07
2014-04-05 07:00:00,CRYPTOCAP:BTC.D,99.441810,99.462343,99.340875,99.460844,1.493888e+07
...,...,...,...,...,...,...
2026-03-07 07:00:00,CRYPTOCAP:BTC.D,59.104914,59.104914,58.927356,58.927356,2.321417e+10
2026-03-08 07:00:00,CRYPTOCAP:BTC.D,58.891095,59.060861,58.660320,58.660320,3.282095e+10
2026-03-09 07:00:00,CRYPTOCAP:BTC.D,58.638598,59.079522,58.638598,59.079522,4.965041e+10


In [14]:

# Chuyển đội định dạng tên cột cho dataframe
BTCD.columns = BTCD.columns.get_level_values(0)
BTCD.reset_index(inplace=True)


In [15]:

# đổi tên cột tránh trùng lặp
BTCD.rename(columns={'close':'BTC Dominance'},inplace=True)
BTCD


,datetime,symbol,open,high,low,BTC Dominance,volume
0,2014-04-01 07:00:00,CRYPTOCAP:BTC.D,99.380551,99.426704,99.296832,99.384652,1.493888e+07
1,2014-04-02 07:00:00,CRYPTOCAP:BTC.D,99.338089,99.433992,99.332664,99.338387,1.493888e+07
2,2014-04-03 07:00:00,CRYPTOCAP:BTC.D,99.339295,99.494693,99.320638,99.399426,1.493888e+07
3,2014-04-04 07:00:00,CRYPTOCAP:BTC.D,99.409834,99.447563,99.384197,99.410501,1.493888e+07
4,2014-04-05 07:00:00,CRYPTOCAP:BTC.D,99.441810,99.462343,99.340875,99.460844,1.493888e+07
...,...,...,...,...,...,...,...
4355,2026-03-07 07:00:00,CRYPTOCAP:BTC.D,59.104914,59.104914,58.927356,58.927356,2.321417e+10
4356,2026-03-08 07:00:00,CRYPTOCAP:BTC.D,58.891095,59.060861,58.660320,58.660320,3.282095e+10
4357,2026-03-09 07:00:00,CRYPTOCAP:BTC.D,58.638598,59.079522,58.638598,59.079522,4.965041e+10
4358,2026-03-10 07:00:00,CRYPTOCAP:BTC.D,59.101852,59.500627,59.101852,59.352780,5.368376e+10


In [16]:

# tao cột date từ datetime để kết hợp với dữ liệu giá
BTCD['Date'] = BTCD['datetime'].dt.date
BTCD


,datetime,symbol,open,high,low,BTC Dominance,volume,Date
0,2014-04-01 07:00:00,CRYPTOCAP:BTC.D,99.380551,99.426704,99.296832,99.384652,1.493888e+07,2014-04-01
1,2014-04-02 07:00:00,CRYPTOCAP:BTC.D,99.338089,99.433992,99.332664,99.338387,1.493888e+07,2014-04-02
2,2014-04-03 07:00:00,CRYPTOCAP:BTC.D,99.339295,99.494693,99.320638,99.399426,1.493888e+07,2014-04-03
3,2014-04-04 07:00:00,CRYPTOCAP:BTC.D,99.409834,99.447563,99.384197,99.410501,1.493888e+07,2014-04-04
4,2014-04-05 07:00:00,CRYPTOCAP:BTC.D,99.441810,99.462343,99.340875,99.460844,1.493888e+07,2014-04-05
...,...,...,...,...,...,...,...,...
4355,2026-03-07 07:00:00,CRYPTOCAP:BTC.D,59.104914,59.104914,58.927356,58.927356,2.321417e+10,2026-03-07
4356,2026-03-08 07:00:00,CRYPTOCAP:BTC.D,58.891095,59.060861,58.660320,58.660320,3.282095e+10,2026-03-08
4357,2026-03-09 07:00:00,CRYPTOCAP:BTC.D,58.638598,59.079522,58.638598,59.079522,4.965041e+10,2026-03-09
4358,2026-03-10 07:00:00,CRYPTOCAP:BTC.D,59.101852,59.500627,59.101852,59.352780,5.368376e+10,2026-03-10


In [17]:

# Kết hợp dữ liệu BTC Dominance vừa thu thập được với dữ liệu giá
df = df.merge(
    BTCD[['Date','BTC Dominance']],
    left_on = 'Date',
    right_on = 'Date',
    how = 'left'
)

df


,timestamp,open,high,low,close,volume,ATR,BB_width_norm,RSI,MACD_Hist,MA20,EMA20,Date,CFGI,BTC Dominance
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,315.640000,NaN,NaN,0.000000,NaN,13529.010000,2018-01-01,NaN,41.779461
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,316.381068,NaN,0.000000,-1.201493,NaN,13527.654699,2018-01-01,NaN,41.779461
2,2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,315.799756,NaN,-0.116553,-1.911172,NaN,13526.833598,2018-01-01,NaN,41.779461
3,2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,315.832221,NaN,-0.190124,-2.312652,NaN,13526.347928,2018-01-01,NaN,41.779461
4,2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,315.745026,NaN,-0.367052,-2.002172,NaN,13526.658373,2018-01-01,NaN,41.779461
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71661,2026-03-10 23:00:00,69721.58,69981.80,69698.46,69948.63,1165.159140,630.619773,0.125565,22715.462812,798.079696,67678.468042,68793.166000,2026-03-10,13.0,59.352780
71662,2026-03-11 00:00:00,69948.64,70173.25,69759.63,70044.87,728.929610,629.331941,0.125669,15875.054066,799.139303,67685.964667,68798.370590,2026-03-11,15.0,59.333411
71663,2026-03-11 01:00:00,70044.87,70144.00,69880.00,70071.26,947.248790,627.163799,0.125801,14663.557223,800.221240,67692.641000,68803.663270,2026-03-11,15.0,59.333411
71664,2026-03-11 02:00:00,70071.26,70119.59,69614.00,69810.72,1612.875480,626.442292,0.125898,61217.518220,800.267129,67698.840000,68807.850616,2026-03-11,15.0,59.333411


In [18]:

# Thu thập dữ liệu total CMC (total crypto market cap) từ nền tảng biểu đồ trading view

# khởi tạo 1 đối tượng của lớp TvDatafeed()
tv = TvDatafeed()

# sử dụng hàm get_hist để lấy dữ liệu lịch sử của total CMC
Total_CMC = tv.get_hist(
    symbol='TOTAL',             # Mã feature mà mình muốn lấy
    exchange='CRYPTOCAP',       # nguồn lấy dữ liệu vốn hóa thị trường trên trading view
    interval=Interval.in_daily, # tần suất lấy dữ liệu theo ngày
    n_bars=5000                 # số lượng dữ liệu cần lấy
)

Total_CMC


,symbol,open,high,low,close,volume
datetime,,,,,,
2014-03-01 07:00:00,CRYPTOCAP:TOTAL,6.918605e+09,7.223891e+09,6.707818e+09,7.116310e+09,4.566425e+07
2014-03-02 07:00:00,CRYPTOCAP:TOTAL,7.135861e+09,7.180783e+09,6.953284e+09,7.059509e+09,1.662461e+07
2014-03-03 07:00:00,CRYPTOCAP:TOTAL,7.110064e+09,8.929191e+09,7.061427e+09,8.515740e+09,1.662461e+07
2014-03-04 07:00:00,CRYPTOCAP:TOTAL,8.515250e+09,8.824169e+09,7.954029e+09,8.451722e+09,1.662461e+07
2014-03-05 07:00:00,CRYPTOCAP:TOTAL,8.459544e+09,8.544210e+09,8.183027e+09,8.410458e+09,1.662461e+07
...,...,...,...,...,...,...
2026-03-07 07:00:00,CRYPTOCAP:TOTAL,2.305366e+12,2.319151e+12,2.270312e+12,2.282732e+12,9.947178e+10
2026-03-08 07:00:00,CRYPTOCAP:TOTAL,2.284177e+12,2.309063e+12,2.234279e+12,2.249293e+12,1.358133e+11
2026-03-09 07:00:00,CRYPTOCAP:TOTAL,2.250182e+12,2.353337e+12,2.244389e+12,2.316883e+12,2.018701e+11


In [19]:

# Chuyển đổi định dạng tên cột cho dataframe
Total_CMC.columns = Total_CMC.columns.get_level_values(0)
Total_CMC.reset_index(inplace=True)
Total_CMC


,datetime,symbol,open,high,low,close,volume
0,2014-03-01 07:00:00,CRYPTOCAP:TOTAL,6.918605e+09,7.223891e+09,6.707818e+09,7.116310e+09,4.566425e+07
1,2014-03-02 07:00:00,CRYPTOCAP:TOTAL,7.135861e+09,7.180783e+09,6.953284e+09,7.059509e+09,1.662461e+07
2,2014-03-03 07:00:00,CRYPTOCAP:TOTAL,7.110064e+09,8.929191e+09,7.061427e+09,8.515740e+09,1.662461e+07
3,2014-03-04 07:00:00,CRYPTOCAP:TOTAL,8.515250e+09,8.824169e+09,7.954029e+09,8.451722e+09,1.662461e+07
4,2014-03-05 07:00:00,CRYPTOCAP:TOTAL,8.459544e+09,8.544210e+09,8.183027e+09,8.410458e+09,1.662461e+07
...,...,...,...,...,...,...,...
4386,2026-03-07 07:00:00,CRYPTOCAP:TOTAL,2.305366e+12,2.319151e+12,2.270312e+12,2.282732e+12,9.947178e+10
4387,2026-03-08 07:00:00,CRYPTOCAP:TOTAL,2.284177e+12,2.309063e+12,2.234279e+12,2.249293e+12,1.358133e+11
4388,2026-03-09 07:00:00,CRYPTOCAP:TOTAL,2.250182e+12,2.353337e+12,2.244389e+12,2.316883e+12,2.018701e+11
4389,2026-03-10 07:00:00,CRYPTOCAP:TOTAL,2.315928e+12,2.412373e+12,2.312989e+12,2.357494e+12,2.134358e+11


In [20]:

# đổi tên cột tránh bị trùng lặp 
Total_CMC.rename(columns={'close':'Total CMC'},inplace=True)
Total_CMC


,datetime,symbol,open,high,low,Total CMC,volume
0,2014-03-01 07:00:00,CRYPTOCAP:TOTAL,6.918605e+09,7.223891e+09,6.707818e+09,7.116310e+09,4.566425e+07
1,2014-03-02 07:00:00,CRYPTOCAP:TOTAL,7.135861e+09,7.180783e+09,6.953284e+09,7.059509e+09,1.662461e+07
2,2014-03-03 07:00:00,CRYPTOCAP:TOTAL,7.110064e+09,8.929191e+09,7.061427e+09,8.515740e+09,1.662461e+07
3,2014-03-04 07:00:00,CRYPTOCAP:TOTAL,8.515250e+09,8.824169e+09,7.954029e+09,8.451722e+09,1.662461e+07
4,2014-03-05 07:00:00,CRYPTOCAP:TOTAL,8.459544e+09,8.544210e+09,8.183027e+09,8.410458e+09,1.662461e+07
...,...,...,...,...,...,...,...
4386,2026-03-07 07:00:00,CRYPTOCAP:TOTAL,2.305366e+12,2.319151e+12,2.270312e+12,2.282732e+12,9.947178e+10
4387,2026-03-08 07:00:00,CRYPTOCAP:TOTAL,2.284177e+12,2.309063e+12,2.234279e+12,2.249293e+12,1.358133e+11
4388,2026-03-09 07:00:00,CRYPTOCAP:TOTAL,2.250182e+12,2.353337e+12,2.244389e+12,2.316883e+12,2.018701e+11
4389,2026-03-10 07:00:00,CRYPTOCAP:TOTAL,2.315928e+12,2.412373e+12,2.312989e+12,2.357494e+12,2.134358e+11


In [21]:

# tạo cột date để kết hợp với dữ liệu giá 
Total_CMC['Date'] = Total_CMC['datetime'].dt.date
Total_CMC


,datetime,symbol,open,high,low,Total CMC,volume,Date
0,2014-03-01 07:00:00,CRYPTOCAP:TOTAL,6.918605e+09,7.223891e+09,6.707818e+09,7.116310e+09,4.566425e+07,2014-03-01
1,2014-03-02 07:00:00,CRYPTOCAP:TOTAL,7.135861e+09,7.180783e+09,6.953284e+09,7.059509e+09,1.662461e+07,2014-03-02
2,2014-03-03 07:00:00,CRYPTOCAP:TOTAL,7.110064e+09,8.929191e+09,7.061427e+09,8.515740e+09,1.662461e+07,2014-03-03
3,2014-03-04 07:00:00,CRYPTOCAP:TOTAL,8.515250e+09,8.824169e+09,7.954029e+09,8.451722e+09,1.662461e+07,2014-03-04
4,2014-03-05 07:00:00,CRYPTOCAP:TOTAL,8.459544e+09,8.544210e+09,8.183027e+09,8.410458e+09,1.662461e+07,2014-03-05
...,...,...,...,...,...,...,...,...
4386,2026-03-07 07:00:00,CRYPTOCAP:TOTAL,2.305366e+12,2.319151e+12,2.270312e+12,2.282732e+12,9.947178e+10,2026-03-07
4387,2026-03-08 07:00:00,CRYPTOCAP:TOTAL,2.284177e+12,2.309063e+12,2.234279e+12,2.249293e+12,1.358133e+11,2026-03-08
4388,2026-03-09 07:00:00,CRYPTOCAP:TOTAL,2.250182e+12,2.353337e+12,2.244389e+12,2.316883e+12,2.018701e+11,2026-03-09
4389,2026-03-10 07:00:00,CRYPTOCAP:TOTAL,2.315928e+12,2.412373e+12,2.312989e+12,2.357494e+12,2.134358e+11,2026-03-10


In [22]:

# kết hợp dữ liệu Total CMC vừa thu thập được với dữ liệu giá ban đầu
df = df.merge(
    Total_CMC[['Date','Total CMC']],
    left_on = 'Date',
    right_on = 'Date',
    how = 'left'
)

df


,timestamp,open,high,low,close,volume,ATR,BB_width_norm,RSI,MACD_Hist,MA20,EMA20,Date,CFGI,BTC Dominance,Total CMC
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,315.640000,NaN,NaN,0.000000,NaN,13529.010000,2018-01-01,NaN,41.779461,5.354920e+11
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,316.381068,NaN,0.000000,-1.201493,NaN,13527.654699,2018-01-01,NaN,41.779461,5.354920e+11
2,2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,315.799756,NaN,-0.116553,-1.911172,NaN,13526.833598,2018-01-01,NaN,41.779461,5.354920e+11
3,2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,315.832221,NaN,-0.190124,-2.312652,NaN,13526.347928,2018-01-01,NaN,41.779461,5.354920e+11
4,2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,315.745026,NaN,-0.367052,-2.002172,NaN,13526.658373,2018-01-01,NaN,41.779461,5.354920e+11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71661,2026-03-10 23:00:00,69721.58,69981.80,69698.46,69948.63,1165.159140,630.619773,0.125565,22715.462812,798.079696,67678.468042,68793.166000,2026-03-10,13.0,59.352780,2.357494e+12
71662,2026-03-11 00:00:00,69948.64,70173.25,69759.63,70044.87,728.929610,629.331941,0.125669,15875.054066,799.139303,67685.964667,68798.370590,2026-03-11,15.0,59.333411,2.350619e+12
71663,2026-03-11 01:00:00,70044.87,70144.00,69880.00,70071.26,947.248790,627.163799,0.125801,14663.557223,800.221240,67692.641000,68803.663270,2026-03-11,15.0,59.333411,2.350619e+12
71664,2026-03-11 02:00:00,70071.26,70119.59,69614.00,69810.72,1612.875480,626.442292,0.125898,61217.518220,800.267129,67698.840000,68807.850616,2026-03-11,15.0,59.333411,2.350619e+12


In [23]:

# Thu thập dữ liệu USDTD (USDT Dominance) từ nền tảng biểu đồ trading view

# khởi tạo 1 đối tượng thuộc lớp TvDatafeed()
tv = TvDatafeed()

# sử dụng hàm get_hist để lấy dữ liệu lịch sử của USDT Dominance
usdt_d = tv.get_hist(
    symbol='USDT.D',            # Mã feature mà mình muốn lấy
    exchange='CRYPTOCAP',       # nguồn lấy dữ liệu vốn hóa thị trường trên trading view
    interval=Interval.in_daily, # tần suất lấy dữ liệu theo ngày
    n_bars=5000                 # số lượng dữ liệu cần lấy
)

usdt_d



,symbol,open,high,low,close,volume
datetime,,,,,,
2015-03-08 07:00:00,CRYPTOCAP:USDT.D,0.006426,0.006562,0.006402,0.006466,4.285012e+04
2015-03-09 07:00:00,CRYPTOCAP:USDT.D,0.006468,0.006491,0.006149,0.006194,4.285012e+04
2015-03-10 07:00:00,CRYPTOCAP:USDT.D,0.006193,0.006282,0.005989,0.006141,4.285012e+04
2015-03-11 07:00:00,CRYPTOCAP:USDT.D,0.006139,0.006235,0.005994,0.006007,4.285012e+04
2015-03-12 07:00:00,CRYPTOCAP:USDT.D,0.006011,0.006165,0.005974,0.006031,4.285012e+04
...,...,...,...,...,...,...
2026-03-07 07:00:00,CRYPTOCAP:USDT.D,7.980097,8.102500,7.933455,8.059219,4.811834e+10
2026-03-08 07:00:00,CRYPTOCAP:USDT.D,8.054122,8.233169,7.968113,8.179033,6.357476e+10
2026-03-09 07:00:00,CRYPTOCAP:USDT.D,8.173096,8.191734,7.816715,7.938592,9.170955e+10


In [24]:

usdt_d.columns = usdt_d.columns.get_level_values(0)
usdt_d.reset_index(inplace=True)
usdt_d


,datetime,symbol,open,high,low,close,volume
0,2015-03-08 07:00:00,CRYPTOCAP:USDT.D,0.006426,0.006562,0.006402,0.006466,4.285012e+04
1,2015-03-09 07:00:00,CRYPTOCAP:USDT.D,0.006468,0.006491,0.006149,0.006194,4.285012e+04
2,2015-03-10 07:00:00,CRYPTOCAP:USDT.D,0.006193,0.006282,0.005989,0.006141,4.285012e+04
3,2015-03-11 07:00:00,CRYPTOCAP:USDT.D,0.006139,0.006235,0.005994,0.006007,4.285012e+04
4,2015-03-12 07:00:00,CRYPTOCAP:USDT.D,0.006011,0.006165,0.005974,0.006031,4.285012e+04
...,...,...,...,...,...,...,...
4017,2026-03-07 07:00:00,CRYPTOCAP:USDT.D,7.980097,8.102500,7.933455,8.059219,4.811834e+10
4018,2026-03-08 07:00:00,CRYPTOCAP:USDT.D,8.054122,8.233169,7.968113,8.179033,6.357476e+10
4019,2026-03-09 07:00:00,CRYPTOCAP:USDT.D,8.173096,8.191734,7.816715,7.938592,9.170955e+10
4020,2026-03-10 07:00:00,CRYPTOCAP:USDT.D,7.942634,7.951056,7.626083,7.801736,9.661587e+10


In [25]:

usdt_d['Date'] = usdt_d['datetime'].dt.date
usdt_d


,datetime,symbol,open,high,low,close,volume,Date
0,2015-03-08 07:00:00,CRYPTOCAP:USDT.D,0.006426,0.006562,0.006402,0.006466,4.285012e+04,2015-03-08
1,2015-03-09 07:00:00,CRYPTOCAP:USDT.D,0.006468,0.006491,0.006149,0.006194,4.285012e+04,2015-03-09
2,2015-03-10 07:00:00,CRYPTOCAP:USDT.D,0.006193,0.006282,0.005989,0.006141,4.285012e+04,2015-03-10
3,2015-03-11 07:00:00,CRYPTOCAP:USDT.D,0.006139,0.006235,0.005994,0.006007,4.285012e+04,2015-03-11
4,2015-03-12 07:00:00,CRYPTOCAP:USDT.D,0.006011,0.006165,0.005974,0.006031,4.285012e+04,2015-03-12
...,...,...,...,...,...,...,...,...
4017,2026-03-07 07:00:00,CRYPTOCAP:USDT.D,7.980097,8.102500,7.933455,8.059219,4.811834e+10,2026-03-07
4018,2026-03-08 07:00:00,CRYPTOCAP:USDT.D,8.054122,8.233169,7.968113,8.179033,6.357476e+10,2026-03-08
4019,2026-03-09 07:00:00,CRYPTOCAP:USDT.D,8.173096,8.191734,7.816715,7.938592,9.170955e+10,2026-03-09
4020,2026-03-10 07:00:00,CRYPTOCAP:USDT.D,7.942634,7.951056,7.626083,7.801736,9.661587e+10,2026-03-10


In [26]:

usdt_d.rename(columns={'close':'USDTD'},inplace=True)
usdt_d


,datetime,symbol,open,high,low,USDTD,volume,Date
0,2015-03-08 07:00:00,CRYPTOCAP:USDT.D,0.006426,0.006562,0.006402,0.006466,4.285012e+04,2015-03-08
1,2015-03-09 07:00:00,CRYPTOCAP:USDT.D,0.006468,0.006491,0.006149,0.006194,4.285012e+04,2015-03-09
2,2015-03-10 07:00:00,CRYPTOCAP:USDT.D,0.006193,0.006282,0.005989,0.006141,4.285012e+04,2015-03-10
3,2015-03-11 07:00:00,CRYPTOCAP:USDT.D,0.006139,0.006235,0.005994,0.006007,4.285012e+04,2015-03-11
4,2015-03-12 07:00:00,CRYPTOCAP:USDT.D,0.006011,0.006165,0.005974,0.006031,4.285012e+04,2015-03-12
...,...,...,...,...,...,...,...,...
4017,2026-03-07 07:00:00,CRYPTOCAP:USDT.D,7.980097,8.102500,7.933455,8.059219,4.811834e+10,2026-03-07
4018,2026-03-08 07:00:00,CRYPTOCAP:USDT.D,8.054122,8.233169,7.968113,8.179033,6.357476e+10,2026-03-08
4019,2026-03-09 07:00:00,CRYPTOCAP:USDT.D,8.173096,8.191734,7.816715,7.938592,9.170955e+10,2026-03-09
4020,2026-03-10 07:00:00,CRYPTOCAP:USDT.D,7.942634,7.951056,7.626083,7.801736,9.661587e+10,2026-03-10


In [27]:

df = df.merge(
    usdt_d[['Date','USDTD']],
    left_on='Date',
    right_on='Date',
    how = 'left'
)

df


,timestamp,open,high,low,close,volume,ATR,BB_width_norm,RSI,MACD_Hist,MA20,EMA20,Date,CFGI,BTC Dominance,Total CMC,USDTD
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,315.640000,NaN,NaN,0.000000,NaN,13529.010000,2018-01-01,NaN,41.779461,5.354920e+11,0.257424
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,316.381068,NaN,0.000000,-1.201493,NaN,13527.654699,2018-01-01,NaN,41.779461,5.354920e+11,0.257424
2,2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,315.799756,NaN,-0.116553,-1.911172,NaN,13526.833598,2018-01-01,NaN,41.779461,5.354920e+11,0.257424
3,2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,315.832221,NaN,-0.190124,-2.312652,NaN,13526.347928,2018-01-01,NaN,41.779461,5.354920e+11,0.257424
4,2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,315.745026,NaN,-0.367052,-2.002172,NaN,13526.658373,2018-01-01,NaN,41.779461,5.354920e+11,0.257424
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71661,2026-03-10 23:00:00,69721.58,69981.80,69698.46,69948.63,1165.159140,630.619773,0.125565,22715.462812,798.079696,67678.468042,68793.166000,2026-03-10,13.0,59.352780,2.357494e+12,7.801736
71662,2026-03-11 00:00:00,69948.64,70173.25,69759.63,70044.87,728.929610,629.331941,0.125669,15875.054066,799.139303,67685.964667,68798.370590,2026-03-11,15.0,59.333411,2.350619e+12,7.823834
71663,2026-03-11 01:00:00,70044.87,70144.00,69880.00,70071.26,947.248790,627.163799,0.125801,14663.557223,800.221240,67692.641000,68803.663270,2026-03-11,15.0,59.333411,2.350619e+12,7.823834
71664,2026-03-11 02:00:00,70071.26,70119.59,69614.00,69810.72,1612.875480,626.442292,0.125898,61217.518220,800.267129,67698.840000,68807.850616,2026-03-11,15.0,59.333411,2.350619e+12,7.823834


In [28]:

# Tính market cap của Bitcoin dựa vào total market cap và BTC dominance
df['Bitcoin CMC'] = (df['BTC Dominance']/100)*df['Total CMC']
df



,timestamp,open,high,low,close,volume,ATR,BB_width_norm,RSI,MACD_Hist,MA20,EMA20,Date,CFGI,BTC Dominance,Total CMC,USDTD,Bitcoin CMC
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,315.640000,NaN,NaN,0.000000,NaN,13529.010000,2018-01-01,NaN,41.779461,5.354920e+11,0.257424,2.237257e+11
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,316.381068,NaN,0.000000,-1.201493,NaN,13527.654699,2018-01-01,NaN,41.779461,5.354920e+11,0.257424,2.237257e+11
2,2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,315.799756,NaN,-0.116553,-1.911172,NaN,13526.833598,2018-01-01,NaN,41.779461,5.354920e+11,0.257424,2.237257e+11
3,2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,315.832221,NaN,-0.190124,-2.312652,NaN,13526.347928,2018-01-01,NaN,41.779461,5.354920e+11,0.257424,2.237257e+11
4,2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,315.745026,NaN,-0.367052,-2.002172,NaN,13526.658373,2018-01-01,NaN,41.779461,5.354920e+11,0.257424,2.237257e+11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71661,2026-03-10 23:00:00,69721.58,69981.80,69698.46,69948.63,1165.159140,630.619773,0.125565,22715.462812,798.079696,67678.468042,68793.166000,2026-03-10,13.0,59.352780,2.357494e+12,7.801736,1.399238e+12
71662,2026-03-11 00:00:00,69948.64,70173.25,69759.63,70044.87,728.929610,629.331941,0.125669,15875.054066,799.139303,67685.964667,68798.370590,2026-03-11,15.0,59.333411,2.350619e+12,7.823834,1.394702e+12
71663,2026-03-11 01:00:00,70044.87,70144.00,69880.00,70071.26,947.248790,627.163799,0.125801,14663.557223,800.221240,67692.641000,68803.663270,2026-03-11,15.0,59.333411,2.350619e+12,7.823834,1.394702e+12
71664,2026-03-11 02:00:00,70071.26,70119.59,69614.00,69810.72,1612.875480,626.442292,0.125898,61217.518220,800.267129,67698.840000,68807.850616,2026-03-11,15.0,59.333411,2.350619e+12,7.823834,1.394702e+12


### Tính target của dự án, độ biến động của Bitcoin (Volatility) 🧮

In [29]:
# Tính toán độ biến động của Bitcoin Volatility

# tính lợi suất của cổ phiếu
data['log_return'] = np.log(df['close']/data['prev_close'])

# tính độ biến động dựa trên giao dịch của 14 ngày (336) gần nhất  
data['Volatility_14'] = data['log_return'].rolling(336).std()

# quy đổi đơn vị của độ biến động sang hệ %/năm để dễ dàng quan sát
df['volatility_14_annual'] = data['Volatility_14'] * np.sqrt(8760)

df

,timestamp,open,high,low,close,volume,ATR,BB_width_norm,RSI,MACD_Hist,MA20,EMA20,Date,CFGI,BTC Dominance,Total CMC,USDTD,Bitcoin CMC,volatility_14_annual
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,315.640000,NaN,NaN,0.000000,NaN,13529.010000,2018-01-01,NaN,41.779461,5.354920e+11,0.257424,2.237257e+11,NaN
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,316.381068,NaN,0.000000,-1.201493,NaN,13527.654699,2018-01-01,NaN,41.779461,5.354920e+11,0.257424,2.237257e+11,NaN
2,2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,315.799756,NaN,-0.116553,-1.911172,NaN,13526.833598,2018-01-01,NaN,41.779461,5.354920e+11,0.257424,2.237257e+11,NaN
3,2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,315.832221,NaN,-0.190124,-2.312652,NaN,13526.347928,2018-01-01,NaN,41.779461,5.354920e+11,0.257424,2.237257e+11,NaN
4,2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,315.745026,NaN,-0.367052,-2.002172,NaN,13526.658373,2018-01-01,NaN,41.779461,5.354920e+11,0.257424,2.237257e+11,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71661,2026-03-10 23:00:00,69721.58,69981.80,69698.46,69948.63,1165.159140,630.619773,0.125565,22715.462812,798.079696,67678.468042,68793.166000,2026-03-10,13.0,59.352780,2.357494e+12,7.801736,1.399238e+12,0.613922
71662,2026-03-11 00:00:00,69948.64,70173.25,69759.63,70044.87,728.929610,629.331941,0.125669,15875.054066,799.139303,67685.964667,68798.370590,2026-03-11,15.0,59.333411,2.350619e+12,7.823834,1.394702e+12,0.613833
71663,2026-03-11 01:00:00,70044.87,70144.00,69880.00,70071.26,947.248790,627.163799,0.125801,14663.557223,800.221240,67692.641000,68803.663270,2026-03-11,15.0,59.333411,2.350619e+12,7.823834,1.394702e+12,0.599838
71664,2026-03-11 02:00:00,70071.26,70119.59,69614.00,69810.72,1612.875480,626.442292,0.125898,61217.518220,800.267129,67698.840000,68807.850616,2026-03-11,15.0,59.333411,2.350619e+12,7.823834,1.394702e+12,0.600167


### Xuất dữ liệu đã thu thập sang file CSV để xử lý 💾

In [ ]:

# Xuất dữ liệu sang file csv
df.to_csv('Bitcoin_time_series_raw.csv')
data.to_csv('btc_technical_indicators_raw.csv')
